#### Imports

In [2]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from lexicalrichness import LexicalRichness

In [4]:
# Load the preprocessed sessions DataFrame from Phase 2
df = pd.read_csv('../data/processed/sessions_clean.csv')

#### Brown's Stage from MLU

In [5]:
def mlu_to_brown_stage(mlu):
    if mlu < 2.0:    return "I"
    elif mlu < 2.5:  return "II"
    elif mlu < 3.0:  return "III"
    elif mlu < 3.75: return "IV"
    else:            return "V"

df['brown_stage'] = df['mlu'].apply(mlu_to_brown_stage)

#### VocD and MATTR

In [6]:
def compute_lex_diversity(text):
    try:
        lex = LexicalRichness(text)
        if lex.words < 50:          # both measures need a minimum sample
            return None, None
        vocd  = lex.vocd()
        mattr = lex.mattr(window_size=50)
        return vocd, mattr
    except:
        return None, None

df[['vocd', 'mattr']] = df['text_with_stops'].apply(
    lambda t: pd.Series(compute_lex_diversity(t))
)

#### word2vec

In [7]:
# Tokenize sessions (use text_with_stops, split into word lists)
tokenized = [text.split() for text in df['text_with_stops']]

# Train word2vec on full corpus (15 dims, following Linke & Ramscar 2025)
w2v = Word2Vec(sentences=tokenized, vector_size=15, window=5,
               min_count=3, workers=4, seed=42)

# Represent each session as mean of its word vectors
def mean_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

session_vecs = np.array([
    mean_vector(text.split(), w2v) for text in df['text_with_stops']
])

# Project to 1D — first PC as the score
pca = PCA(n_components=1)
df['w2v_score'] = pca.fit_transform(session_vecs).flatten()

#### Export

In [8]:
rrl_cols = ['file_id', 'child_id', 'age_months',
            'mlu', 'brown_stage', 'vocd', 'mattr', 'w2v_score']

df[rrl_cols].to_csv("../data/processed/pipeline_rrl.csv", index=False)
print(df[rrl_cols].describe().round(3))

       file_id  age_months      mlu     vocd    mattr  w2v_score
count  214.000     214.000  214.000  214.000  214.000    214.000
mean   106.500      41.189    2.762   67.798    0.609     -0.000
std     61.921      11.238    0.940   17.722    0.067      0.598
min      0.000      18.000    0.790   15.905    0.372     -1.548
25%     53.250      32.040    2.140   56.658    0.566     -0.315
50%    106.500      40.735    2.835   69.375    0.624      0.200
75%    159.750      50.720    3.402   81.130    0.660      0.456
max    213.000      62.400    4.929  104.769    0.734      1.071
